# SageMaker AI 환경 정리

<div class="alert alert-warning"> 이 노트북은 <code>SageMaker Distribution Image 3.4.2</code>를 사용하는 SageMaker Studio JupyterLab 인스턴스에서 SageMaker Python SDK 버전 <code>2.251.1</code>로 마지막 테스트되었습니다</div>

<div style="border: 4px solid coral; text-align: center; margin: auto;">
    <p style=" text-align: center; margin: auto;">
        <b>이 노트북은 사용자 환경에서 실행한 모든 노트북이 생성한 모든 리소스를 제거합니다.</b>
    </p>
</div>

❗ 다음 코드 셀은:
- Studio 환경에서 프로비저닝한 프로젝트를 영구적으로 삭제합니다
- 피처 그룹(Feature Group)을 영구적으로 삭제합니다
- 프로젝트가 프로비저닝한 S3 버킷을 영구적으로 삭제합니다
- 프로젝트 관련 접두사 아래의 S3 버킷 객체를 영구적으로 삭제합니다
- 추론 엔드포인트를 영구적으로 삭제합니다

<div class="alert alert-info"> ❗ 프로젝트, 피처 그룹, 엔드포인트 목록이 있는 코드 셀은 <b>이 워크샵 외부에서</b> 생성된 것을 포함하여 AWS 계정의 모든 SageMaker 리소스를 표시합니다. 자신의 리소스를 삭제하지 않도록 삭제할 리소스의 이름을 다시 확인하세요.
</div>

<div class="alert alert-info"><strong> 이 노트북은 AWS 계정의 리소스를 영구적으로 삭제합니다. 삭제할 리소스의 이름을 다시 한 번 확인하세요! </strong>
</div>

<div class="alert alert-info">강사 주도 워크샵에서 AWS가 프로비저닝한 AWS 계정을 사용하는 경우 이 노트북을 실행할 필요가 없습니다.</div>

<div class="alert alert-info"> 이 노트북에는 JupyterLab에서 <code>Python 3</code> 커널을 사용하고 있는지 확인하세요.</div>

In [ ]:
import sagemaker
import boto3
import time
import json
import os

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[오류] 00-start-here 노트북을 실행해야 합니다   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
sm = boto3.client("sagemaker")
s3 = boto3.resource('s3')

## 프로젝트 삭제

In [ ]:
# 현재 도메인에서 생성된 모든 프로젝트 가져오기
projects = [
    {"ProjectName":p["ProjectName"], "ProjectId":p["ProjectId"]} for p in sm.list_projects(MaxResults=100, SortBy="CreationTime")["ProjectSummaryList"] 
        if sm.describe_project(ProjectName=p["ProjectName"])["CreatedBy"]["DomainId"] == domain_id and p["ProjectStatus"] == "CreateCompleted"
]

print(f"도메인 {domain_id}에서 생성된 프로젝트: {json.dumps(projects, indent=2)}")

In [ ]:
# 삭제할 프로젝트 선택
projects_to_delete = []

for p in projects:
    print(f"이 프로젝트를 삭제하시겠습니까: {p['ProjectName']}? (y/n)")
    choice = input()
    if choice == 'y':
        projects_to_delete.append(p)
        
print(f"***************************************")
print(f"다음 프로젝트가 삭제됩니다:\n{json.dumps(projects_to_delete, indent=2)}")
print(f"***************************************")

In [ ]:
for p in projects_to_delete:
    try:
        print(f"프로젝트 삭제 중 {p['ProjectName']}:{sm.delete_project(ProjectName=p['ProjectName'])}")
    except Exception:
        pass

## 모니터링 스케줄 삭제
MLOps 프로젝트에 의해 배포된 엔드포인트를 성공적으로 제거하려면 모니터링 스케줄을 삭제해야 합니다.

In [ ]:
def delete_mon_schedules(endpoint_name, sm_client):
    print(f"엔드포인트의 모니터링 스케줄 삭제: {endpoint_name}")
    for s in sm_client.list_monitoring_schedules(EndpointName=endpoint_name)["MonitoringScheduleSummaries"]:
        print(f"모니터링 스케줄 삭제: {s['MonitoringScheduleName']}")
        r = sm_client.delete_monitoring_schedule(MonitoringScheduleName=s['MonitoringScheduleName'])
        print(r)

In [ ]:
for p in projects_to_delete:
    delete_mon_schedules(f"{p['ProjectName']}-staging", sm)
    delete_mon_schedules(f"{p['ProjectName']}-prod", sm)

## CloudFormation 스택 삭제
이 섹션은 프로젝트가 생성한 AWS CloudFormation 스택을 삭제합니다

In [ ]:
cfn = boto3.client("cloudformation")

for p in projects_to_delete:
    for s in [
            f"sagemaker-{p['ProjectName']}-{p['ProjectId']}-deploy-staging",
            f"sagemaker-{p['ProjectName']}-{p['ProjectId']}-deploy-prod"
            ]:
        try:
            print(f"CloudFormation 스택 삭제: {s}")
            r = cfn.delete_stack(StackName=s)
            print(r)
            time.sleep(180)
        except Exception as e:
            print(f"{s} 삭제 중 예외 발생:{e}")
            pass

## 피처 그룹 삭제

In [ ]:
feature_groups = sm.list_feature_groups(
    FeatureGroupStatusEquals="Created", 
    SortOrder="Descending", 
    SortBy="CreationTime"
)["FeatureGroupSummaries"]

In [ ]:
feature_groups

In [ ]:
# 삭제할 피처 그룹 선택
feature_groups_to_delete = []

for fg in feature_groups:
    print(f"이 피처 그룹을 삭제하시겠습니까: {fg['FeatureGroupName']}? (y/n)")
    choice = input()
    if choice == 'y':
        feature_groups_to_delete.append(fg["FeatureGroupName"])
        
print(f"********************************************")
print(f"다음 피처 그룹이 삭제됩니다:\n{json.dumps(feature_groups_to_delete, indent=2)}")
print(f"********************************************")

In [ ]:
def delete_offline_store(feature_group_name: str):
    try:
        offline_store_config = sm.describe_feature_group(FeatureGroupName=feature_group_name)['OfflineStoreConfig']

    except Exception:
        print(f'피처 그룹: {feature_group_name}에는 오프라인 스토어가 없습니다!')
        return
    
    offline_store_s3_uri = offline_store_config['S3StorageConfig']['ResolvedOutputS3Uri']
    print(f"{offline_store_s3_uri} 아래의 모든 피처 스토어 객체가 삭제됩니다!")
    print("이러한 객체를 삭제하시겠습니까? (y/n)")
    
    choice = input()
    if choice == 'y':
        !aws s3 rm {offline_store_s3_uri} --recursive

<div class="alert alert-info"> 💡 <strong> 다음 코드 셀은 선택한 피처 그룹을 삭제합니다!</strong>
</div>

In [ ]:
for fg in feature_groups_to_delete:
    print(f"피처 그룹 삭제 중: {fg}")
    delete_offline_store(fg)
    sm.delete_feature_group(FeatureGroupName=fg)

## 프로젝트가 프로비저닝한 S3 버킷 삭제

<div class="alert alert-info"> 💡 <strong> 다음 코드 셀은 프로젝트가 생성한 모든 S3 버킷을 삭제합니다!</strong> 강사 주도 워크샵에 참여 중이고 프로비저닝된 AWS 계정을 사용하는 경우 S3 버킷을 삭제하지 못할 수 있습니다. 여기서 중지할 수 있습니다.
</div>

In [ ]:
print(f"*****************************************************")
print(f"다음 S3 버킷이 영구적으로 제거됩니다!")
print(f"*****************************************************")
for p in projects_to_delete:
    print(f"sagemaker-project-{p['ProjectId']}")

In [ ]:
for p in projects_to_delete:
    !aws s3 rb s3://sagemaker-project-{p['ProjectId']} --force 

## 추론 엔드포인트 제거

In [ ]:
endpoints = sm.list_endpoints()["Endpoints"]

In [ ]:
endpoints_to_delete = []

for ep in endpoints:
    print(f"이 엔드포인트를 삭제하시겠습니까: {ep['EndpointName']}? (y/n)")
    choice = input()
    if choice == 'y':
        endpoints_to_delete.append(ep['EndpointName'])
        
print(f"*********** 다음 엔드포인트가 삭제됩니다 ***********")
print('\n'.join(endpoints_to_delete))
print(f"*******************************************************")

In [ ]:
for ep in endpoints_to_delete:
    try:
        delete_mon_schedules(ep, sm)
        time.sleep(10)
        print(f"엔드포인트 삭제 중: {ep}:{sm.delete_endpoint(EndpointName=ep)}")
    except Exception as e:
        print(f"{ep} 삭제 중 예외 발생:{e}")
        pass

## SageMaker S3 데이터 버킷에서 프로젝트 관련 객체 제거

<div class="alert alert-info"> 💡 <strong> 다음 코드 셀은 지정된 S3 접두사 아래의 모든 객체를 삭제합니다!</strong>
</div>

In [ ]:
prefixes_to_delete = [
    bucket_prefix,
    #s3_fs_query_output_prefix
]

In [ ]:
print(f"************************************************************************")
print(f"다음 S3 접두사 아래의 모든 객체가 영구적으로 제거됩니다!")
print(f"************************************************************************")
for p in prefixes_to_delete:
    print(f"{bucket_name}/{p}")

<div class="alert alert-info"> ❗ <strong> S3 접두사를 다시 확인하세요! 이 접두사 아래의 모든 데이터가 영구적으로 삭제됩니다!</strong>
</div>

In [ ]:
# rm 명령의 주석을 해제하세요
for p in prefixes_to_delete:
    # !aws s3 rm s3://{bucket_name}/{p} --recursive
    pass

## MLflow 서버 삭제
MLflow 추적 서버는 생성되면 삭제하거나 중지할 때까지 비용이 발생합니다. 추적 서버에 대한 요금은 서버가 실행된 기간, 선택한 크기 및 추적 서버에 기록된 데이터 양을 기준으로 청구됩니다.

추가 비용을 피하려면 수동으로 생성한 모든 MLFlow 추적 서버를 삭제해야 합니다.

<div class="alert alert-info">UX 또는 API를 통해 수동으로 추적 서버를 생성하지 않았다면 MLflow 추적 서버를 삭제하고 해당 셀을 실행할 필요가 없습니다.</div>

In [ ]:
# 도메인에서 활성화된 모든 MLflow 서버 찾기
tracking_servers = sm.list_mlflow_tracking_servers(
    TrackingServerStatus='Created',
)['TrackingServerSummaries']

ts_to_delete = []

for ts in tracking_servers:
    print(f"이 MLflow 추적 서버를 삭제하시겠습니까: {ts['TrackingServerName']}? (y/n)")
    choice = input()
    if choice == 'y':
        ts_to_delete.append(ts['TrackingServerName'])

print(f"*********** 다음 MLFLOW 추적 서버가 삭제됩니다 ***********")
print('\n'.join(ts_to_delete))
print(f"**********************************************************************")

In [ ]:
for ts in ts_to_delete:
    try:
        print(f"MLflow 추적 서버 삭제 중: {ts}:{sm.delete_mlflow_tracking_server(ts)}")
    except Exception as e:
        print(f"{ts} 삭제 중 예외 발생:{e}")
        pass

# 커널 종료

In [ ]:
%%html

<p><b>리소스를 해제하기 위해 이 노트북의 커널을 종료합니다.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>